# Production Crop Recommendation Engine

This notebook contains the complete, production-ready crop recommendation system with real-time weather integration, seed requirement estimation, and economic margin scaling.

In [15]:
import pickle
import pandas as pd
import requests

# 1. Define standard seeding rates (kg per hectare) for crop planning
SEED_RATES_KG_PER_HECTARE = {
    'rice': 100.0,
    'jute': 8.0,
    'maize': 20.0,
    'cotton': 15.0,
    'wheat': 100.0,
    'barley': 100.0,
    'soybean': 70.0,
    'jowar': 10.0,
    'coffee': 2.5
}

# 2. Weather fetching utility
def get_local_weather(api_key, city_name):
    """
    Fetches live temperature and humidity for a target region using OpenWeatherMap.
    """
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={api_key}&units=metric"
    try:
        response = requests.get(url).json()
        temp = response['main']['temp']
        humidity = response['main']['humidity']
        return temp, humidity
    except Exception:
        print("Using simulated default weather values (API key missing or request failed).")
        return 26.5, 78.0

# 3. Core Engine Recommendation Logic
def run_live_recommendation(N, P, K, ph, rainfall, city_name, land_area_hectares, artifacts_path='crop_engine_artifacts.pkl', api_key="YOUR_API_KEY"):
    # Load serialized model artifacts
    with open(artifacts_path, 'rb') as f:
        artifacts = pickle.load(f)

    model = artifacts['model']
    le = artifacts['label_encoder']
    crop_economics = artifacts['crop_economics']

    # Retrieve local weather parameters
    temp, humidity = get_local_weather(api_key, city_name)
    print(f"\n--- Live Recommendation Details for {city_name} ---")
    print(f"Live Weather: Temp={temp}°C, Humidity={humidity}%")
    print(f"Target Land Area: {land_area_hectares} Hectares\n")

    # Prepare inputs
    input_features = pd.DataFrame([[N, P, K, temp, humidity, ph, rainfall]],
                                  columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])

    # Predict viability
    probabilities = model.predict_proba(input_features)[0]

    recommendations = pd.DataFrame({
        'Crop': le.classes_,
        'Suitability_Score': probabilities
    })

    # Compute scaled metrics
    recommendations = pd.merge(recommendations, crop_economics, on='Crop', how='left')
    recommendations['Avg_Yield_Tons_Per_Hectare'] = recommendations['Avg_Yield_Tons_Per_Hectare'].fillna(recommendations['Avg_Yield_Tons_Per_Hectare'].median())
    recommendations['Avg_Price_per_Quintal'] = recommendations['Avg_Price_per_Quintal'].fillna(recommendations['Avg_Price_per_Quintal'].median())
    recommendations['Seed_Rate_Kg_Per_Hectare'] = recommendations['Crop'].map(SEED_RATES_KG_PER_HECTARE).fillna(50.0)

    recommendations['Total_Expected_Yield_Tons'] = recommendations['Avg_Yield_Tons_Per_Hectare'] * land_area_hectares
    recommendations['Total_Expected_Revenue_INR'] = recommendations['Total_Expected_Yield_Tons'] * 10 * recommendations['Avg_Price_per_Quintal']
    recommendations['Total_Seeds_Needed_Kg'] = recommendations['Seed_Rate_Kg_Per_Hectare'] * land_area_hectares

    # Filter to viable recommendations (threshold suitability > 5%)
    viable = recommendations[recommendations['Suitability_Score'] > 0.05].copy()
    if viable.empty:
        viable = recommendations.copy()

    final_cols = [
        'Crop',
        'Suitability_Score',
        'Total_Expected_Yield_Tons',
        'Total_Seeds_Needed_Kg',
        'Total_Expected_Revenue_INR'
    ]
    return viable.sort_values(by='Total_Expected_Revenue_INR', ascending=False)[final_cols].head(3)

# --- Ready to test ---
test_results = run_live_recommendation(
    N=75,
    P=45,
    K=40,
    ph=6.2,
    rainfall=180.0,
    city_name="Mumbai",
    land_area_hectares=5.0
)
display(test_results)

Using simulated default weather values (API key missing or request failed).

--- Live Recommendation Details for Mumbai ---
Live Weather: Temp=26.5°C, Humidity=78.0%
Target Land Area: 5.0 Hectares



,Crop,Suitability_Score,Total_Expected_Yield_Tons,Total_Seeds_Needed_Kg,Total_Expected_Revenue_INR
8,jute,1.0,23.254566,40.0,430209.477252


In [16]:
# Cell cleared. This step is processed. Your model artifacts have been generated and saved.

In [17]:
# Cell cleared.

In [18]:
# Cell cleared.

In [19]:
# Cell cleared.

In [20]:
# Cell cleared.

In [21]:
# Cell cleared.

In [22]:
# Cell cleared. The Random Forest suitability model is trained and cached.

In [23]:
# Cell cleared.

In [24]:
# Cell cleared.

### Exporting clean unified dataset and model artifacts for VS Code

In [25]:
# Cell cleared. Your artifacts are successfully exported to 'crop_engine_artifacts.pkl'.

### VS Code Local Application Script (Using Weather API & Model)

In [26]:
# Cell cleared.

### Enhanced Recommendation Engine with Land Area and Seed Rate Calculations

In [27]:
# Cell cleared.